# Claims Reconciliation — Business & Data Quality Validation

## Purpose

This notebook validates data consistency between the **Policies** and **Claims** datasets in the Silver layer before building business KPIs in the Gold layer.

The main objective is to verify whether the number of claims reported in the Policies dataset matches the actual number of claim records available in the Claims dataset.

## Why is this important?

- **Data Consistency:** Identify discrepancies between reported claim counts and available claim records.
- **Data Quality:** Detect policies with missing or unmatched claim records.
- **KPI Accuracy:** Ensure that Claim Frequency, Average Claim Severity, and Pure Premium are calculated using appropriate data sources.
- **Business Reliability:** Prevent misleading analytics caused by incomplete claim information.

## Expected Outcome

The notebook produces reconciliation metrics, identifies mismatched policies, and highlights data limitations that must be considered when designing Gold-layer KPIs.

**Note:** This notebook performs validation and analysis only. It does not modify the Silver tables.

In [0]:
# Import PySpark functions
from pyspark.sql import functions as F

# Load validated policy and claim datasets
df_policies = spark.table(
    "auto_insurance.silver.policies"
)

df_claims = spark.table(
    "auto_insurance.silver.claims"
)

df_orphans = spark.table(
    "auto_insurance.silver.claims_orphan"
)

# Display source record counts
print("Policies:", df_policies.count())
print("Valid claims:", df_claims.count())
print("Orphan claims:", df_orphans.count())

Policies: 678013
Valid claims: 26444
Orphan claims: 195


In [0]:
# Calculate the total number of claims reported in policy records
reported_claims = (
    df_policies
    .agg(F.sum("claim_count").alias("total"))
    .first()["total"]
)

# Count individual claim records with matching policies
recorded_claims = df_claims.count()

# Calculate the difference between both sources
claim_difference = reported_claims - recorded_claims

print("Claims reported in Policies:", reported_claims)
print("Claims recorded in Claims:", recorded_claims)
print("Difference:", claim_difference)

Claims reported in Policies: 36102
Claims recorded in Claims: 26444
Difference: 9658


In [0]:
# Aggregate claim records at policy level
df_claims_aggregated = (
    df_claims
    .groupBy("policy_id")
    .agg(
        F.count("*").alias("recorded_claim_count"),
        F.sum("claim_amount").alias("total_claim_amount")
    )
)

# Join reported claim counts with recorded claim counts
df_reconciliation = (
    df_policies
    .select("policy_id", "claim_count")
    .join(
        df_claims_aggregated,
        on="policy_id",
        how="left"
    )
    .fillna({
        "recorded_claim_count": 0,
        "total_claim_amount": 0.0
    })
    .withColumn(
        "claim_count_difference",
        F.col("claim_count") - F.col("recorded_claim_count")
    )
)

# Summarize reconciliation results
display(
    df_reconciliation.agg(
        F.count("*").alias("total_policies"),
        F.sum(
            F.when(F.col("claim_count_difference") == 0, 1)
             .otherwise(0)
        ).alias("matching_policies"),
        F.sum(
            F.when(F.col("claim_count_difference") != 0, 1)
             .otherwise(0)
        ).alias("mismatching_policies"),
        F.sum(
            F.when(F.col("claim_count_difference") > 0, 1)
             .otherwise(0)
        ).alias("policies_with_missing_claim_records"),
        F.sum(
            F.when(F.col("claim_count_difference") < 0, 1)
             .otherwise(0)
        ).alias("policies_with_extra_claim_records")
    )
)

total_policies,matching_policies,mismatching_policies,policies_with_missing_claim_records,policies_with_extra_claim_records
678013,668896,9117,9117,0


In [0]:
# Identify policies with differences between reported and recorded claims
df_mismatches = (
    df_reconciliation
    .filter(F.col("claim_count_difference") != 0)
    .orderBy(F.desc(F.abs(F.col("claim_count_difference"))))
)

# Display a sample of policies requiring investigation
display(df_mismatches.limit(20))

policy_id,claim_count,recorded_claim_count,total_claim_amount,claim_count_difference
19471,11,0,0.0,11
24495,4,0,0.0,4
1147,4,0,0.0,4
19656,3,0,0.0,3
1653,3,0,0.0,3
9972,3,0,0.0,3
5889,3,0,0.0,3
7059,3,0,0.0,3
21321,3,0,0.0,3
6973,3,0,0.0,3
